# 2.1 对话系统开发 - ConversationManager

构建智能知识库Agent的对话引擎, 支持多轮对话、上下文管理、记忆压缩

## 项目定位

本实验的 ConversationManager 将被6.3最终项目直接使用

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client, verify_config
from agent_project import *
from agent_project.tools import create_default_registry

client = get_client()
print(f"项目模块已加载 | LLM: {client.name} ({client.model})")


In [ ]:
# 创建对话管理器
from agent_project.conversation import ConversationManager

conv = ConversationManager(
    system_prompt="你是智能知识库助手。基于知识库回答问题。",
    max_history=20,
    max_context_tokens=4000,
)
print(f"ConversationManager已创建: max_history={conv.max_history}, max_tokens={conv.max_context_tokens}")


In [ ]:
# 模拟多轮对话
print("===== 多轮对话测试 =====\n")

# 第1轮: 用户提问
conv.add_user_message("什么是RAG?")
r = client.chat(messages=conv.get_context(), temperature=0.3, max_tokens=200)
conv.add_assistant_message(r["content"])
print(f"[第1轮] 用户: 什么是RAG?")
print(f"        助手: {r['content'][:100]}...")
print(f"        上下文: {conv.turn_count}轮, ~{conv._token_count}tokens\n")

# 第2轮: 追问 (依赖上下文)
conv.add_user_message("那它和普通搜索有什么区别?")
r = client.chat(messages=conv.get_context(), temperature=0.3, max_tokens=200)
conv.add_assistant_message(r["content"])
print(f"[第2轮] 用户: 那它和普通搜索有什么区别?")
print(f"        助手: {r['content'][:100]}...")
print(f"        上下文: {conv.turn_count}轮\n")

# 第3轮: 继续追问
conv.add_user_message("给出一个实际应用的例子")
r = client.chat(messages=conv.get_context(), temperature=0.3, max_tokens=200)
conv.add_assistant_message(r["content"])
print(f"[第3轮] 用户: 给出一个实际应用的例子")
print(f"        助手: {r['content'][:100]}...")
print(f"        上下文: {conv.turn_count}轮\n")


In [ ]:
# 对话历史压缩测试
print("===== 历史压缩测试 =====\n")
print(f"压缩前: {conv.turn_count}轮, ~{conv._token_count}tokens")
conv._compress_history()
print(f"压缩后: {len(conv.messages)}条消息, ~{conv._token_count}tokens")
if conv.summary:
    print(f"摘要: {conv.summary[:200]}...")


In [ ]:
# 流式输出演示
print("===== 流式输出 =====\n")
print("助手(流式): ", end="")
for token in client.chat_stream("用50字介绍智能知识库Agent的工作原理"):
    print(token, end="", flush=True)
print()


In [ ]:
# 导出对话状态
state = conv.to_dict()
print(f"会话状态: {state['turn_count']}轮, {state['token_count']}tokens")
print(f"模块就绪: ConversationManager 可被6.3最终项目导入使用")
